In [1]:
import pandas as pd
import numpy as np
   
import torch
import torch.nn as nn
from torch.optim import Adam
import torch.nn.functional as F

from Model import Model

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from mlflow.models import infer_signature

from logger import logger

log = logger("Wine quality")

2025/05/03 08:01:50 INFO mlflow.tracking.fluent: Experiment with name 'Wine quality' does not exist. Creating a new experiment.


In [2]:
data=pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [3]:
X = data.drop(columns='quality').to_numpy()
y = data['quality'].to_numpy()


# X = X.astype(np.float64)
# y = y.astype(np.int64)

signature = infer_signature(X, y)



mu = np.mean(X, axis = 0)
sd = np.sqrt(np.var(X, axis = 0))


X = (X - mu)/sd

# X = torch.from_numpy(X.values)
# y = torch.from_numpy(y.values)

In [4]:
X = torch.tensor(X, dtype=torch.float32)  # Features as FloatTensor
y = torch.tensor(y, dtype=torch.long)  


X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3)




In [5]:
def accuracy(output, target):

    # Get predicted class by taking argmax
    preds = torch.argmax(F.sigmoid(output), dim=1)
    
    # Compare predictions to ground truth
    correct = (preds == target).sum().item()
    total = target.size(0)

    return correct / total

def train_model(params):
    model = Model(input_dim=params['input_dim'],
                  output_dim=params['output_dim'],
                  hidden_dims=params['hidden_dims'])
    lr = params['lr']
    epochs = params['epochs']
    optimizer = Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for e in range(epochs):
        optimizer.zero_grad()
        pred = model(X_train)
        loss = criterion(pred, y_train)
        loss.backward()
        optimizer.step()
        if e % 20 == 0:
            acc = accuracy(pred, y_train)
            print(f"Accuracy is {acc}")
        
    # test case
    with torch.no_grad():
        pred = model(X_test)
        loss = criterion(pred, y_test)
        print(f"Final Loss is {loss.detach().cpu()}")
        acc = accuracy(pred, y_test)
        print(f"Test Accuracy is {acc}")
        
    return loss.detach().cpu(), acc, model
    
        

In [9]:
log.start_run()
params = {'input_dim': 11,
          'output_dim': 11,
          'hidden_dims': [32,64,32,16],
          'lr': 0.01,
          'epochs' : 1000}


loss, acc, model = train_model(params)


log.log_param(params)
log.log_metrics({'Test Loss': loss,
                 'Test Accuracy': acc})
log.create_artifacts(signature, model)
log.end_run()

Accuracy is 0.1516919486581097
Accuracy is 0.49620770128354724
Accuracy is 0.5399649941656943
Accuracy is 0.5755542590431738
Accuracy is 0.5933488914819136
Accuracy is 0.5971411901983664
Accuracy is 0.6085180863477246
Accuracy is 0.6277712952158693
Accuracy is 0.646732788798133
Accuracy is 0.6645274212368728
Accuracy is 0.6866977829638273
Accuracy is 0.6980746791131855
Accuracy is 0.7027421236872812
Accuracy is 0.7109101516919487
Accuracy is 0.7091598599766628
Accuracy is 0.7292882147024504
Accuracy is 0.7450408401400234
Accuracy is 0.7570011668611435
Accuracy is 0.7441656942823804
Accuracy is 0.7494165694282381
Accuracy is 0.7826721120186698
Accuracy is 0.7698366394399067
Accuracy is 0.7896732788798133
Accuracy is 0.7599183197199533
Accuracy is 0.8013418903150525
Accuracy is 0.7771295215869312
Accuracy is 0.7718786464410735
Accuracy is 0.8133022170361727
Accuracy is 0.7231621936989499
Accuracy is 0.8063010501750292
Accuracy is 0.822928821470245
Accuracy is 0.8028004667444574
Accuracy 

2025/05/03 08:04:12 WARNING mlflow.utils.requirements_utils: Found torch version (2.7.0+cu118) contains a local version label (+cu118). MLflow logged a pip requirement for this package as 'torch==2.7.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Accuracy is 0.8652275379229871
Accuracy is 0.8378063010501751
Final Loss is 2.3636910915374756
Test Accuracy is 0.5612244897959183


2025/05/03 08:04:16 WARNING mlflow.utils.requirements_utils: Found torch version (2.7.0+cu118) contains a local version label (+cu118). MLflow logged a pip requirement for this package as 'torch==2.7.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


🏃 View run classy-stag-999 at: http://127.0.0.1:5000/#/experiments/215977801730706845/runs/29fb9d8d9d8b427ca83544174b4c988f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/215977801730706845
